In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import urllib.request

from model import Iteration_Model 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
url = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

with urllib.request.urlopen(req) as response:
    raw_text = response.read().decode("utf-8")

# Strip the Project Gutenberg header and footer
start_idx = raw_text.find("I. A SCANDAL IN BOHEMIA")
end_idx = raw_text.find("End of the Project Gutenberg")
text = raw_text[start_idx:end_idx].strip()

print(f"Loaded {len(text)} characters of normal English text.")
print(f"Sample preview:\n{text[:200]}")

Loaded 592375 characters of normal English text.
Sample preview:
I. A SCANDAL IN BOHEMIA


I.

To Sherlock Holmes she is always _the_ woman. I have seldom heard him
mention her under any other name. In his eyes she eclipses and
predominates the whole of her 


In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for i, ch in enumerate(chars)}

def encode(string):
    return [char_to_id[ch] for ch in string]

def decode(indices):
    return "".join([id_to_char[idx] for idx in indices])

# Convert entire book into a single tensor
data = torch.tensor(encode(text), dtype=torch.long, device=device)
print(f"Vocab size: {vocab_size} unique characters")

Vocab size: 97 unique characters


In [4]:
def get_batch(batch_size=32, seq_len=64):
    max_start = len(data) - seq_len - 1
    start_indices = torch.randint(0, max_start, (batch_size,))
    
    inputs = torch.stack([data[i:i + seq_len] for i in start_indices])
    targets = torch.stack([data[i + 1:i + seq_len + 1] for i in start_indices])
    
    return inputs, targets

In [5]:
def generate_sample(model, prompt="Sherlock Holmes was ", length=120, temperature=0.8):
    model.eval()
    tokens = encode(prompt)
    curr = torch.tensor([tokens], dtype=torch.long, device=device)
    
    with torch.no_grad():
        for _ in range(length):
            # Crop to the last 64 characters if the string gets longer than max_seq_len
            context = curr[:, -64:]
            
            logits = model(context)
            last_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(last_logits, dim=-1)
            
            next_idx = torch.multinomial(probs, num_samples=1).item()
            next_tensor = torch.tensor([[next_idx]], dtype=torch.long, device=device)
            curr = torch.cat([curr, next_tensor], dim=1)
            
    model.train()
    return decode(curr[0].tolist())

In [9]:
model = Iteration_Model(
    vocab_size=vocab_size,
    hidden_dim=128,
    max_seq_len=128,
    num_layers=4,
    sweep_iters=4,
    layer_iters=3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

num_steps = 10000

for step in range(1, num_steps + 1):
    inputs, targets = get_batch(batch_size=32, seq_len=64)
    
    optimizer.zero_grad()
    predictions = model(inputs)
    
    loss = criterion(predictions.reshape(-1, vocab_size), targets.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    if step % 100 == 0 or step == 1:
        print(f"Step {step:04d} | Loss {loss.item():.4f}")
        sample = generate_sample(model, prompt="The man was ", length=80)
        print(f"Sample: {repr(sample)}\n")

Step 0001 | Loss 5.9694
Sample: 'The man was      i  d eiatw   e t   MwtT/e   i Ttd eeyeWwt id  whi     h    e   d T    T yy '

Step 0100 | Loss 2.5411
Sample: 'The man was andatt\r\n\r\ncoul dathat hedtatate hesosanthe tsion thathas p t t hace tacere me ke'

Step 0200 | Loss 2.4155
Sample: 'The man was ow.\r\n\r\n“Yon thin soup,” ruves win che then uch a noup mytous whet to ay in hapud'

Step 0300 | Loss 2.0619
Sample: 'The man was lumees seatinme troniverest tillyesteredf doown whill weas the\r\nand I the shaglm'

Step 0400 | Loss 1.9936
Sample: 'The man was ben thimpe.”\r\n\r\n“Holmes.”\r\n\r\n“We whone it sere tel, of house, of my coulf I dor '

Step 0500 | Loss 1.8908
Sample: 'The man was noor the dorica, and your Pandy what fer writh\r\nfrom ark, and a your have be y h'

Step 0600 | Loss 1.8018
Sample: 'The man was signging mach up the facter fater costry to the dingsidly and\r\ndright the was ac'

Step 0700 | Loss 1.7352
Sample: 'The man was out a pend. He wold se,\r\nsuright it

In [7]:
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"{name:25s} | grad norm: {param.grad.norm().item():.8f}")

embedding.weight          | grad norm: 0.01148399
pos_embedding.weight      | grad norm: 0.00708863
layers.0.F.weight         | grad norm: 0.08588450
layers.0.F.bias           | grad norm: 0.02014547
layers.0.L.weight         | grad norm: 0.09043179
layers.0.L.bias           | grad norm: 0.02014547
layers.0.B.weight         | grad norm: 0.07015429
layers.0.B.bias           | grad norm: 0.02014547
layers.0.FLB_Attention.W_q.weight | grad norm: 0.05507661
layers.0.FLB_Attention.W_q.bias | grad norm: 0.00726289
layers.0.FLB_Attention.W_k.weight | grad norm: 0.03192290
layers.0.FLB_Attention.W_k.bias | grad norm: 0.00000000
layers.0.FLB_Attention.W_v.weight | grad norm: 0.13687825
layers.0.FLB_Attention.W_v.bias | grad norm: 0.02991791
layers.0.norm_fwd.weight  | grad norm: 0.00728997
layers.0.norm_fwd.bias    | grad norm: 0.01998215
layers.0.norm_lat.weight  | grad norm: 0.00468434
layers.0.norm_lat.bias    | grad norm: 0.01305421
layers.0.norm_bck.weight  | grad norm: 0.00336051
layers.0